# Pipeline de Unión de Contaminantes, Clima e Índices Urbanos AppEEARS

Este *notebook* implementa el proceso de integración entre los *datasets* horarios de contaminantes y clima, ya generados previamente, y los índices urbanos obtenidos mediante AppEEARS:

1. Índice de vegetación, NDVI.
2. Índice de edificación/asfalto, NDBI.

El objetivo es obtener un único *dataset* por ciudad con la misma granularidad temporal que los registros de contaminantes y clima. Como dichos registros se encuentran a nivel horario, mientras que los índices satelitales se observan cada varios días, es necesario aplicar un proceso de interpolación temporal. En concreto,

- Los contaminantes y variables climáticas están en resolución horaria.
- El NDVI procede de composiciones satelitales de 16 días.
- El NDBI procede de composiciones satelitales de 8 días.

Por tanto, no se puede hacer una unión directa mediante fecha exacta, ya que habría valores de NDVI y NDBI solo en algunos instantes concretos. Para resolverlo, hacemos uso de la interpolación PCHIP, siglas de ***Piecewise Cubic Hermite Interpolating Polynomial***.

### **Método de interpolación PCHIP**

La interpolación PCHIP consiste en construir una función cúbica por tramos entre cada par de observaciones consecutivas. Es decir, en lugar de unir dos puntos mediante una recta, como ocurre en la interpolación lineal, PCHIP utiliza polinomios de grado tres. Esto permite generar una transición más suave entre valores consecutivos. Sin embargo, a diferencia de una interpolación cúbica clásica o de un *spline* cúbico convencional, PCHIP está diseñado para conservar la forma de los datos originales. Esto significa que intenta evitar oscilaciones artificiales, sobrepasamientos y valores extremos que no estaban sugeridos por las observaciones reales.

Esta diferencia es importante. Una interpolación lineal sería sencilla y estable, pero produciría una evolución a tramos rectos, con cambios bruscos de pendiente en cada fecha satelital. En variables ambientales de evolución lenta, como la vegetación o la superficie construida, esta representación puede resultar demasiado rígida. Por el contrario, una interpolación cúbica clásica puede ser más suave, pero también puede introducir ondulaciones artificiales entre puntos, especialmente cuando hay cambios locales en la tendencia. Esto podría generar valores poco realistas de `NDVI` o `NDBI`.

La lógica seguida es:

1. Para cada ciudad, se extrae la serie temporal original del índice.
2. Se ordena temporalmente.
3. Se interpola la serie sobre la malla horaria del dataset de contaminantes y clima.
4. Se acotan los valores al rango físico esperable de los índices, [-1, 1].
5. Se unen las nuevas columnas NDVI y NDBI al dataset horario de cada ciudad.

In [8]:
# =============================================================================
# IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN DE RUTAS
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# Mostramos todas las columnas cuando visualicemos dataframes.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# =============================================================================
# IMPORTACIÓN DEL MÉTODO DE INTERPOLACIÓN PCHIP
# =============================================================================

# PCHIP será el método principal de interpolación.
from scipy.interpolate import PchipInterpolator

In [9]:
# =============================================================================
# RUTAS DE LOS DATASETS
# =============================================================================

BASE_PATH = Path("..", "..") / "datasets"
FOLDER_CONT_CLIMA = BASE_PATH / "archivos_cont_clima"
FOLDER_APPEEARS = BASE_PATH / "AppEEARS"
PATH_NDVI = FOLDER_APPEEARS / "índice_vegetación.csv"
PATH_NDBI = FOLDER_APPEEARS / "índice_edificación.csv"
FOLDER_FINAL = BASE_PATH / "archivos_cont_clima_indices"
FOLDER_FINAL.mkdir(parents=True, exist_ok=True)

In [10]:
# =============================================================================
# FUNCIÓN DE LECTURA ROBUSTA DE CSV
# =============================================================================

def leer_csv_robusto(ruta, nrows=None):
    """
    Lee un archivo CSV intentando detectar automáticamente el separador.

    En este proyecto hay archivos que pueden venir separados por:
    - coma ,
    - punto y coma ;

    Además, algunos archivos se guardan con encoding utf-8-sig para que Excel
    reconozca correctamente tildes y caracteres especiales.
    """

    ruta = Path(ruta)

    try:
        df = pd.read_csv(
            ruta,
            sep=None,
            engine="python",
            encoding="utf-8-sig",
            nrows=nrows
        )
        return df

    except Exception as e1:
        print(f"⚠️ Falló lectura automática para {ruta.name}: {e1}")
        
        try:
            df = pd.read_csv(
                ruta,
                sep=";",
                encoding="utf-8-sig",
                nrows=nrows
            )
            return df
        
        except Exception as e2:
            print(f"⚠️ Falló lectura con ';' para {ruta.name}: {e2}")
            
            df = pd.read_csv(
                ruta,
                sep=",",
                encoding="utf-8-sig",
                nrows=nrows
            )
            return df

In [11]:
# =============================================================================
# PARÁMETROS GENERALES DEL PROCESO
# =============================================================================

# Periodo temporal común.
# Aunque aparecen registros desde 2009, los índices AppEEARS están
# pensados para años posteriores. Por eso restringimos el dataset final a este intervalo.
FECHA_INICIO = pd.Timestamp("2013-01-01 00:00:00")
FECHA_FIN = pd.Timestamp("2024-12-31 23:00:00")

# Método de interpolación PCHIP.
METODO_INTERPOLACION = "pchip"

# Rango físico esperado de los índices.
# NDVI y NDBI suelen interpretarse en el intervalo [-1, 1].
MIN_INDICE = -1
MAX_INDICE = 1

# Decimales para guardar NDVI y NDBI.
DECIMALES_INDICES = 6

# Separador de salida.
SEPARADOR_SALIDA = ","

print("⚙️ PARÁMETROS DEL PROCESO")
print("-" * 100)
print(f"Fecha inicio: {FECHA_INICIO}")
print(f"Fecha fin: {FECHA_FIN}")
print(f"Método de interpolación: {METODO_INTERPOLACION}")
print(f"Rango índices: [{MIN_INDICE}, {MAX_INDICE}]")

⚙️ PARÁMETROS DEL PROCESO
----------------------------------------------------------------------------------------------------
Fecha inicio: 2013-01-01 00:00:00
Fecha fin: 2024-12-31 23:00:00
Método de interpolación: pchip
Rango índices: [-1, 1]


In [12]:
# =============================================================================
# CARGA DE LOS ÍNDICES APPEEARS
# =============================================================================

# Leemos el índice de vegetación.
df_ndvi = leer_csv_robusto(PATH_NDVI)

# Leemos el índice de edificación/asfalto.
df_ndbi = leer_csv_robusto(PATH_NDBI)

print("✅ Archivos de índices cargados correctamente.")

print("\n🌱 NDVI")
print("-" * 100)
print(df_ndvi.shape)
print(df_ndvi.columns.tolist())
display(df_ndvi.head(10))

print("\n🏙️ NDBI")
print("-" * 100)
print(df_ndbi.shape)
print(df_ndbi.columns.tolist())
display(df_ndbi.head(10))

✅ Archivos de índices cargados correctamente.

🌱 NDVI
----------------------------------------------------------------------------------------------------
(3324, 3)
['Ciudad', 'Fecha', 'NDVI']


,Ciudad,Fecha,NDVI
0,A Corua,2012-12-18,0.000046
1,A Corua,2013-01-01,0.000036
2,A Corua,2013-01-17,0.000043
3,A Corua,2013-02-02,0.000043
4,A Corua,2013-02-18,0.000043
5,A Corua,2013-03-06,0.000053
6,A Corua,2013-03-22,0.000063
7,A Corua,2013-04-07,0.000048
8,A Corua,2013-04-23,0.000037
9,A Corua,2013-05-09,0.000046



🏙️ NDBI
----------------------------------------------------------------------------------------------------
(6624, 3)
['Ciudad', 'Fecha', 'NDBI']


,Ciudad,Fecha,NDBI
0,A Corua,2012-12-26,-0.163860
1,A Corua,2013-01-01,-0.159821
2,A Corua,2013-01-09,-0.169538
3,A Corua,2013-01-17,-0.210487
4,A Corua,2013-01-25,-0.208741
5,A Corua,2013-02-02,-0.158702
6,A Corua,2013-02-10,-0.135284
7,A Corua,2013-02-18,-0.132712
8,A Corua,2013-02-26,-0.169898
9,A Corua,2013-03-06,-0.143117


In [13]:
# =============================================================================
# PREPARACIÓN DE LOS ÍNDICES
# =============================================================================

# Los archivos ya vienen limpios con las columnas:
# Ciudad | Fecha | NDVI
# Ciudad | Fecha | NDBI
#
# Por tanto, aquí solo hacemos lo estrictamente necesario:
# - convertir Fecha a datetime,
# - convertir el índice a numérico,
# - ordenar por ciudad y fecha,
# - eliminar filas sin fecha o sin valor.

# -----------------------------------------------------------------------------
# NDVI
# -----------------------------------------------------------------------------

df_ndvi["Fecha"] = pd.to_datetime(df_ndvi["Fecha"], errors="coerce")
df_ndvi["NDVI"] = pd.to_numeric(df_ndvi["NDVI"], errors="coerce")

df_ndvi = df_ndvi.dropna(subset=["Ciudad", "Fecha", "NDVI"])
df_ndvi = df_ndvi.sort_values(["Ciudad", "Fecha"])

# Si por alguna razón hay varias observaciones de NDVI para la misma ciudad y fecha,
# se calcula la media. Esto evita problemas al interpolar.
df_ndvi = (
    df_ndvi
    .groupby(["Ciudad", "Fecha"], as_index=False)["NDVI"]
    .mean()
    .sort_values(["Ciudad", "Fecha"])
)


# -----------------------------------------------------------------------------
# NDBI
# -----------------------------------------------------------------------------

df_ndbi["Fecha"] = pd.to_datetime(df_ndbi["Fecha"], errors="coerce")
df_ndbi["NDBI"] = pd.to_numeric(df_ndbi["NDBI"], errors="coerce")

df_ndbi = df_ndbi.dropna(subset=["Ciudad", "Fecha", "NDBI"])
df_ndbi = df_ndbi.sort_values(["Ciudad", "Fecha"])

# Si hay duplicados ciudad-fecha, se promedian.
df_ndbi = (
    df_ndbi
    .groupby(["Ciudad", "Fecha"], as_index=False)["NDBI"]
    .mean()
    .sort_values(["Ciudad", "Fecha"])
)

print("✅ Fechas y valores preparados.")

print("\nResumen NDVI:")
display(df_ndvi.groupby("Ciudad")["NDVI"].agg(["count", "min", "max", "mean"]))

print("\nResumen NDBI:")
display(df_ndbi.groupby("Ciudad")["NDBI"].agg(["count", "min", "max", "mean"]))

✅ Fechas y valores preparados.

Resumen NDVI:


,count,min,max,mean
Ciudad,,,,
A Corua,277,0.000018,0.000080,0.000044
Albacete,277,0.000010,0.000020,0.000014
AlicanteAlacant,277,0.000010,0.000022,0.000014
Barcelona,277,0.000005,0.000018,0.000013
Bilbao,277,0.000010,0.000069,0.000019
Madrid,277,0.000008,0.000016,0.000011
Murcia,277,0.000011,0.000025,0.000017
Santa Cruz de Tenerife,277,0.000011,0.000031,0.000016
Sevilla,277,0.000009,0.000021,0.000014



Resumen NDBI:


,count,min,max,mean
Ciudad,,,,
A Corua,552,-0.271348,0.061814,-0.131889
Albacete,552,-0.121054,0.165504,0.113387
AlicanteAlacant,552,-0.073402,0.117092,0.051613
Barcelona,552,-0.087216,0.155809,0.097164
Bilbao,552,-0.256339,0.149769,0.052148
Madrid,552,-0.265945,0.178348,0.111881
Murcia,552,-0.018276,0.100425,0.052906
Santa Cruz de Tenerife,552,-0.136933,0.080645,0.005639
Sevilla,552,-0.114234,0.123637,0.068375


In [14]:
# =============================================================================
# CONTROL DE ESCALA DEL NDVI
# =============================================================================

# En las primeras filas aparece:
# Madrid | 2013-01-01 | NDVI = 0.000010
#
# Esto es demasiado pequeño para un NDVI ya escalado a [0, 1].
# Lo normal sería encontrar valores del orden de 0.05, 0.1, 0.2, etc.,
# especialmente si se trata de un punto urbano o periurbano.
#
# Por tanto, si todos los valores absolutos del NDVI son menores que 0.01,
# interpretamos que el índice se ha dividido entre 10000 una vez de más.
#
# En ese caso multiplicamos por 10000 para recuperar una escala interpretable.
# Ejemplo:
# 0.000010 * 10000 = 0.10

max_abs_ndvi = df_ndvi["NDVI"].abs().max()

print("🔎 Control de escala NDVI")
print("-" * 100)
print(f"Máximo valor absoluto de NDVI antes del control: {max_abs_ndvi}")

if max_abs_ndvi < 0.01:
    print("⚠️ Los valores de NDVI parecen estar en una escala demasiado pequeña.")
    print("   Se multiplican por 10000 para corregir un posible doble escalado.")
    
    df_ndvi["NDVI"] = df_ndvi["NDVI"] * 10000

else:
    print("✅ La escala del NDVI parece razonable. No se modifica.")

# Acotamos igualmente al rango físico [-1, 1].
df_ndvi["NDVI"] = df_ndvi["NDVI"].clip(MIN_INDICE, MAX_INDICE)

print("\nResumen NDVI después del control:")
display(df_ndvi.groupby("Ciudad")["NDVI"].agg(["count", "min", "max", "mean"]))

🔎 Control de escala NDVI
----------------------------------------------------------------------------------------------------
Máximo valor absoluto de NDVI antes del control: 8.035e-05
⚠️ Los valores de NDVI parecen estar en una escala demasiado pequeña.
   Se multiplican por 10000 para corregir un posible doble escalado.

Resumen NDVI después del control:


,count,min,max,mean
Ciudad,,,,
A Corua,277,0.1840,0.8035,0.438504
Albacete,277,0.0954,0.2017,0.136390
AlicanteAlacant,277,0.1028,0.2162,0.144627
Barcelona,277,0.0484,0.1827,0.128360
Bilbao,277,0.0985,0.6879,0.187665
Madrid,277,0.0775,0.1580,0.112342
Murcia,277,0.1094,0.2469,0.172043
Santa Cruz de Tenerife,277,0.1074,0.3052,0.160154
Sevilla,277,0.0884,0.2105,0.141564


In [15]:
# =============================================================================
# CONTROL DE ESCALA DEL NDBI
# =============================================================================

# En las primeras filas de Madrid, el NDBI aparece con valores como 0.1.
# Esa escala sí parece razonable.
#
# Aun así, acotamos los valores al intervalo [-1, 1] por seguridad.

print("🔎 Control de escala NDBI")
print("-" * 100)

print("Resumen NDBI antes de acotar:")
display(df_ndbi.groupby("Ciudad")["NDBI"].agg(["count", "min", "max", "mean"]))

df_ndbi["NDBI"] = df_ndbi["NDBI"].clip(MIN_INDICE, MAX_INDICE)

print("\nResumen NDBI después de acotar:")
display(df_ndbi.groupby("Ciudad")["NDBI"].agg(["count", "min", "max", "mean"]))

🔎 Control de escala NDBI
----------------------------------------------------------------------------------------------------
Resumen NDBI antes de acotar:


,count,min,max,mean
Ciudad,,,,
A Corua,552,-0.271348,0.061814,-0.131889
Albacete,552,-0.121054,0.165504,0.113387
AlicanteAlacant,552,-0.073402,0.117092,0.051613
Barcelona,552,-0.087216,0.155809,0.097164
Bilbao,552,-0.256339,0.149769,0.052148
Madrid,552,-0.265945,0.178348,0.111881
Murcia,552,-0.018276,0.100425,0.052906
Santa Cruz de Tenerife,552,-0.136933,0.080645,0.005639
Sevilla,552,-0.114234,0.123637,0.068375



Resumen NDBI después de acotar:


,count,min,max,mean
Ciudad,,,,
A Corua,552,-0.271348,0.061814,-0.131889
Albacete,552,-0.121054,0.165504,0.113387
AlicanteAlacant,552,-0.073402,0.117092,0.051613
Barcelona,552,-0.087216,0.155809,0.097164
Bilbao,552,-0.256339,0.149769,0.052148
Madrid,552,-0.265945,0.178348,0.111881
Murcia,552,-0.018276,0.100425,0.052906
Santa Cruz de Tenerife,552,-0.136933,0.080645,0.005639
Sevilla,552,-0.114234,0.123637,0.068375


In [16]:
# =============================================================================
# CORRECCIÓN ESPECÍFICA DE NOMBRES DE CIUDAD EN LOS ÍNDICES
# =============================================================================

# En los archivos de AppEEARS hay algunos nombres que no coinciden exactamente
# con los nombres oficiales que estamos usando en el resto del proyecto.
#
# En concreto:
# - "A Corua" aparece sin la ñ.
# - "AlicanteAlacant" aparece sin barra entre los dos nombres.
#
# Como la unión se hace por igualdad exacta de strings, estas diferencias provocan
# que no se encuentren observaciones de NDVI ni de NDBI para esas ciudades.

CORRECCION_NOMBRES_INDICES = {
    "A Corua": "A Coruña",
    "A Coruna": "A Coruña",
    "A CoruÃ±a": "A Coruña",
    "AlicanteAlacant": "Alicante/Alacant",
    "Alicante Alacant": "Alicante/Alacant",
    "Alicante-Alacant": "Alicante/Alacant"
}

# Aplicamos la corrección tanto al índice de vegetación como al de edificación.
df_ndvi["Ciudad"] = df_ndvi["Ciudad"].replace(CORRECCION_NOMBRES_INDICES)
df_ndbi["Ciudad"] = df_ndbi["Ciudad"].replace(CORRECCION_NOMBRES_INDICES)

print("✅ Corrección de nombres aplicada.")

print("\n🌱 Ciudades en NDVI después de corregir:")
print("-" * 80)
print(sorted(df_ndvi["Ciudad"].unique()))

print("\n🏙️ Ciudades en NDBI después de corregir:")
print("-" * 80)
print(sorted(df_ndbi["Ciudad"].unique()))

✅ Corrección de nombres aplicada.

🌱 Ciudades en NDVI después de corregir:
--------------------------------------------------------------------------------
['A Coruña', 'Albacete', 'Alicante/Alacant', 'Barcelona', 'Bilbao', 'Madrid', 'Murcia', 'Santa Cruz de Tenerife', 'Sevilla', 'Valencia', 'Valladolid', 'Zaragoza']

🏙️ Ciudades en NDBI después de corregir:
--------------------------------------------------------------------------------
['A Coruña', 'Albacete', 'Alicante/Alacant', 'Barcelona', 'Bilbao', 'Madrid', 'Murcia', 'Santa Cruz de Tenerife', 'Sevilla', 'Valencia', 'Valladolid', 'Zaragoza']


In [17]:
# =============================================================================
# MAPEO ENTRE NOMBRES DE ARCHIVO Y NOMBRES DE CIUDAD
# =============================================================================

# En archivos_cont_clima, algunos nombres vienen adaptados al sistema de archivos:
# - A_Coruña.csv
# - Santa_Cruz_de_Tenerife.csv
# - Alicante-Alacant.csv
#
# En los índices, las ciudades aparecen en la columna Ciudad con nombres normales.
# Para evitar errores, definimos el mapeo explícitamente.

MAPEO_CIUDADES = {
    "A_Coruña": "A Coruña",
    "Albacete": "Albacete",
    "Alicante-Alacant": "Alicante/Alacant",
    "Barcelona": "Barcelona",
    "Bilbao": "Bilbao",
    "Madrid": "Madrid",
    "Murcia": "Murcia",
    "Santa_Cruz_de_Tenerife": "Santa Cruz de Tenerife",
    "Sevilla": "Sevilla",
    "Valencia": "Valencia",
    "Valladolid": "Valladolid",
    "Zaragoza": "Zaragoza"
}

# Listamos los archivos reales.
archivos_cont_clima = sorted(FOLDER_CONT_CLIMA.glob("*.csv"))

print("📌 Archivos encontrados en archivos_cont_clima")
print("-" * 100)
for archivo in archivos_cont_clima:
    print(f"- {archivo.name}")

# Comprobamos que todos los archivos tengan mapeo.
print("\n🔎 Comprobación de mapeo")
print("-" * 100)

for archivo in archivos_cont_clima:
    nombre_archivo = archivo.stem
    
    if nombre_archivo in MAPEO_CIUDADES:
        print(f"✅ {nombre_archivo} -> {MAPEO_CIUDADES[nombre_archivo]}")
    else:
        print(f"❌ Falta mapeo para: {nombre_archivo}")

📌 Archivos encontrados en archivos_cont_clima
----------------------------------------------------------------------------------------------------
- A_Coruña.csv
- Albacete.csv
- Alicante-Alacant.csv
- Barcelona.csv
- Bilbao.csv
- Madrid.csv
- Murcia.csv
- Santa_Cruz_de_Tenerife.csv
- Sevilla.csv
- Valencia.csv
- Valladolid.csv
- Zaragoza.csv

🔎 Comprobación de mapeo
----------------------------------------------------------------------------------------------------
✅ A_Coruña -> A Coruña
✅ Albacete -> Albacete
✅ Alicante-Alacant -> Alicante/Alacant
✅ Barcelona -> Barcelona
✅ Bilbao -> Bilbao
✅ Madrid -> Madrid
✅ Murcia -> Murcia
✅ Santa_Cruz_de_Tenerife -> Santa Cruz de Tenerife
✅ Sevilla -> Sevilla
✅ Valencia -> Valencia
✅ Valladolid -> Valladolid
✅ Zaragoza -> Zaragoza


In [18]:
# =============================================================================
# COMPROBACIÓN DE DISPONIBILIDAD DE NDVI Y NDBI POR CIUDAD
# =============================================================================

# Antes de interpolar, comprobamos que cada ciudad de archivos_cont_clima
# tenga observaciones en los dos índices.

resumen_disponibilidad = []

for archivo in archivos_cont_clima:
    
    ciudad_archivo = archivo.stem
    ciudad_indice = MAPEO_CIUDADES.get(ciudad_archivo)
    
    if ciudad_indice is None:
        resumen_disponibilidad.append({
            "Archivo": archivo.name,
            "Ciudad_indices": None,
            "NDVI_obs": 0,
            "NDBI_obs": 0,
            "Estado": "Sin mapeo"
        })
        continue
    
    n_ndvi = len(df_ndvi[df_ndvi["Ciudad"] == ciudad_indice])
    n_ndbi = len(df_ndbi[df_ndbi["Ciudad"] == ciudad_indice])
    
    estado = "OK" if n_ndvi > 0 and n_ndbi > 0 else "Revisar"
    
    resumen_disponibilidad.append({
        "Archivo": archivo.name,
        "Ciudad_indices": ciudad_indice,
        "NDVI_obs": n_ndvi,
        "NDBI_obs": n_ndbi,
        "Estado": estado
    })

df_disponibilidad = pd.DataFrame(resumen_disponibilidad)

display(df_disponibilidad)

,Archivo,Ciudad_indices,NDVI_obs,NDBI_obs,Estado
0,A_Coruña.csv,A Coruña,277,552,OK
1,Albacete.csv,Albacete,277,552,OK
2,Alicante-Alacant.csv,Alicante/Alacant,277,552,OK
3,Barcelona.csv,Barcelona,277,552,OK
4,Bilbao.csv,Bilbao,277,552,OK
5,Madrid.csv,Madrid,277,552,OK
6,Murcia.csv,Murcia,277,552,OK
7,Santa_Cruz_de_Tenerife.csv,Santa Cruz de Tenerife,277,552,OK
8,Sevilla.csv,Sevilla,277,552,OK
9,Valencia.csv,Valencia,277,552,OK


In [19]:
# =============================================================================
# INTERPOLACIÓN HORARIA Y UNIÓN CON LOS DATASETS DE CONTAMINANTES + CLIMA
# =============================================================================

# En esta celda se realiza el proceso principal:
#
# Para cada ciudad:
# 1. Se carga el CSV horario de contaminantes + clima.
# 2. Se convierte Start y End a datetime.
# 3. Se filtra al periodo común 2013-2024.
# 4. Se interpola NDVI sobre las horas del dataset.
# 5. Se interpola NDBI sobre las horas del dataset.
# 6. Se añaden ambas columnas al dataset.
# 7. Se guarda el nuevo CSV en archivos_cont_clima_indices.

resumen_final = []

for archivo in tqdm(archivos_cont_clima, desc="Interpolando índices y uniendo datasets"):
    
    print("\n" + "=" * 120)
    print(f"Procesando archivo: {archivo.name}")
    print("=" * 120)
    
    # -------------------------------------------------------------------------
    # 1. Identificación de la ciudad
    # -------------------------------------------------------------------------
    
    ciudad_archivo = archivo.stem
    ciudad_indice = MAPEO_CIUDADES.get(ciudad_archivo)
    
    if ciudad_indice is None:
        print(f"❌ No hay mapeo definido para {ciudad_archivo}. Se omite.")
        continue
    
    print(f"Ciudad en archivo: {ciudad_archivo}")
    print(f"Ciudad en índices: {ciudad_indice}")
    
    
    # -------------------------------------------------------------------------
    # 2. Carga del dataset horario de contaminantes + clima
    # -------------------------------------------------------------------------
    
    df_base = leer_csv_robusto(archivo)
    
    # Convertimos las fechas principales.
    df_base["Start"] = pd.to_datetime(df_base["Start"], errors="coerce")
    df_base["End"] = pd.to_datetime(df_base["End"], errors="coerce")
    
    # Eliminamos filas sin Start, porque no se podrían ordenar ni unir.
    df_base = df_base.dropna(subset=["Start"])
    
    # Ordenamos temporalmente.
    df_base = df_base.sort_values("Start")
    
    
    # -------------------------------------------------------------------------
    # 3. Filtrado al periodo temporal común
    # -------------------------------------------------------------------------
    
    # Esto es importante porque AppEEARS se está usando para 2013-2024.
    # Si mantuviéramos 2009-2012, habría que extrapolar NDVI y NDBI.
    # Eso metodológicamente es más débil, así que filtramos al rango común.
    df_base = df_base[
        (df_base["Start"] >= FECHA_INICIO) &
        (df_base["Start"] <= FECHA_FIN)
    ].copy()
    
    if df_base.empty:
        print("❌ El dataset queda vacío tras filtrar el periodo común. Se omite.")
        continue
    
    # Si hubiera horas duplicadas, consolidamos promediando numéricas.
    # Normalmente no debería ocurrir, pero lo dejamos por seguridad.
    if df_base["Start"].duplicated().any():
        print("⚠️ Hay horas duplicadas. Se agrupan por Start.")
        
        columnas_numericas = df_base.select_dtypes(include=["number"]).columns.tolist()
        columnas_no_numericas = [
            c for c in df_base.columns
            if c not in columnas_numericas and c != "Start"
        ]
        
        agg_dict = {c: "mean" for c in columnas_numericas}
        agg_dict.update({c: "first" for c in columnas_no_numericas})
        
        df_base = (
            df_base
            .groupby("Start", as_index=False)
            .agg(agg_dict)
            .sort_values("Start")
        )
    
    # Timeline horario objetivo.
    # Usamos las propias horas del dataset base, que ya es el eje principal.
    timeline = pd.DatetimeIndex(df_base["Start"]).sort_values()
    
    
    # -------------------------------------------------------------------------
    # 4. Preparación de las series NDVI y NDBI de esa ciudad
    # -------------------------------------------------------------------------
    
    serie_ndvi = (
        df_ndvi[df_ndvi["Ciudad"] == ciudad_indice]
        .groupby("Fecha")["NDVI"]
        .mean()
        .sort_index()
    )
    
    serie_ndbi = (
        df_ndbi[df_ndbi["Ciudad"] == ciudad_indice]
        .groupby("Fecha")["NDBI"]
        .mean()
        .sort_index()
    )
    
    if serie_ndvi.empty:
        print("⚠️ No hay datos de NDVI para esta ciudad.")
    
    if serie_ndbi.empty:
        print("⚠️ No hay datos de NDBI para esta ciudad.")
    
    
    # -------------------------------------------------------------------------
    # 5. Interpolación NDVI
    # -------------------------------------------------------------------------
    
    if serie_ndvi.empty:
        
        # Si no hubiera datos, se rellena con NaN.
        df_base["NDVI"] = np.nan
        metodo_ndvi = "sin datos"
    
    elif len(serie_ndvi) == 1:
        
        # Con un único dato no se puede interpolar.
        # Se asigna constante.
        df_base["NDVI"] = float(serie_ndvi.iloc[0])
        metodo_ndvi = "constante por único dato"
    
    else:
        
        # Convertimos fechas a números para interpolar.
        # Usamos días desde el inicio del periodo común.
        x_obs = ((serie_ndvi.index - FECHA_INICIO) / pd.Timedelta(days=1)).astype(float)
        y_obs = serie_ndvi.values.astype(float)
        
        x_target = ((timeline - FECHA_INICIO) / pd.Timedelta(days=1)).astype(float)
        
        if METODO_INTERPOLACION == "pchip" and len(serie_ndvi) >= 3:
            
            # PCHIP sin extrapolación.
            interpolador_ndvi = PchipInterpolator(
                x_obs,
                y_obs,
                extrapolate=False
            )
            
            valores_ndvi = interpolador_ndvi(x_target)
            
            # En los extremos, si queda algún NaN por estar fuera del rango exacto
            # de observación satelital, usamos el valor observado más cercano.
            valores_ndvi = (
                pd.Series(valores_ndvi, index=timeline)
                .ffill()
                .bfill()
                .values
            )
            
            metodo_ndvi = "PCHIP"
        
        else:
            
            # Alternativa lineal.
            valores_ndvi = np.interp(
                x_target,
                x_obs,
                y_obs
            )
            
            metodo_ndvi = "lineal"
        
        # Acotamos al rango físico.
        valores_ndvi = np.clip(valores_ndvi, MIN_INDICE, MAX_INDICE)
        
        df_base["NDVI"] = valores_ndvi
    
    
    # -------------------------------------------------------------------------
    # 6. Interpolación NDBI
    # -------------------------------------------------------------------------
    
    if serie_ndbi.empty:
        
        df_base["NDBI"] = np.nan
        metodo_ndbi = "sin datos"
    
    elif len(serie_ndbi) == 1:
        
        df_base["NDBI"] = float(serie_ndbi.iloc[0])
        metodo_ndbi = "constante por único dato"
    
    else:
        
        x_obs = ((serie_ndbi.index - FECHA_INICIO) / pd.Timedelta(days=1)).astype(float)
        y_obs = serie_ndbi.values.astype(float)
        
        x_target = ((timeline - FECHA_INICIO) / pd.Timedelta(days=1)).astype(float)
        
        if METODO_INTERPOLACION == "pchip" and len(serie_ndbi) >= 3:
            
            interpolador_ndbi = PchipInterpolator(
                x_obs,
                y_obs,
                extrapolate=False
            )
            
            valores_ndbi = interpolador_ndbi(x_target)
            
            valores_ndbi = (
                pd.Series(valores_ndbi, index=timeline)
                .ffill()
                .bfill()
                .values
            )
            
            metodo_ndbi = "PCHIP"
        
        else:
            
            valores_ndbi = np.interp(
                x_target,
                x_obs,
                y_obs
            )
            
            metodo_ndbi = "lineal"
        
        valores_ndbi = np.clip(valores_ndbi, MIN_INDICE, MAX_INDICE)
        
        df_base["NDBI"] = valores_ndbi
    
    
    # -------------------------------------------------------------------------
    # 7. Redondeo final de índices
    # -------------------------------------------------------------------------
    
    df_base["NDVI"] = df_base["NDVI"].round(DECIMALES_INDICES)
    df_base["NDBI"] = df_base["NDBI"].round(DECIMALES_INDICES)
    
    
    # -------------------------------------------------------------------------
    # 8. Guardado del dataset final
    # -------------------------------------------------------------------------
    
    ruta_salida = FOLDER_FINAL / archivo.name
    
    df_base.to_csv(
        ruta_salida,
        index=False,
        encoding="utf-8-sig",
        sep=SEPARADOR_SALIDA
    )
    
    
    # -------------------------------------------------------------------------
    # 9. Registro diagnóstico
    # -------------------------------------------------------------------------
    
    resumen_final.append({
        "Ciudad_archivo": ciudad_archivo,
        "Ciudad_indices": ciudad_indice,
        "Filas_finales": len(df_base),
        "Fecha_inicio_final": df_base["Start"].min(),
        "Fecha_fin_final": df_base["Start"].max(),
        "NDVI_obs_originales": len(serie_ndvi),
        "NDBI_obs_originales": len(serie_ndbi),
        "Metodo_NDVI": metodo_ndvi,
        "Metodo_NDBI": metodo_ndbi,
        "NDVI_min": df_base["NDVI"].min(),
        "NDVI_max": df_base["NDVI"].max(),
        "NDBI_min": df_base["NDBI"].min(),
        "NDBI_max": df_base["NDBI"].max(),
        "NDVI_NaN": df_base["NDVI"].isna().sum(),
        "NDBI_NaN": df_base["NDBI"].isna().sum(),
        "Ruta_salida": str(ruta_salida)
    })
    
    print(f"✅ Guardado: {ruta_salida}")
    print(f"   Filas finales: {len(df_base)}")
    print(f"   Método NDVI: {metodo_ndvi}")
    print(f"   Método NDBI: {metodo_ndbi}")
    print(f"   Rango NDVI: [{df_base['NDVI'].min()}, {df_base['NDVI'].max()}]")
    print(f"   Rango NDBI: [{df_base['NDBI'].min()}, {df_base['NDBI'].max()}]")

Interpolando índices y uniendo datasets:   0%|          | 0/12 [00:00<?, ?it/s]


Procesando archivo: A_Coruña.csv
Ciudad en archivo: A_Coruña
Ciudad en índices: A Coruña


Interpolando índices y uniendo datasets:   8%|▊         | 1/12 [00:05<00:56,  5.10s/it]

✅ Guardado: ..\..\datasets\archivos_cont_clima_indices\A_Coruña.csv
   Filas finales: 105192
   Método NDVI: PCHIP
   Método NDBI: PCHIP
   Rango NDVI: [0.184, 0.8035]
   Rango NDBI: [-0.271348, 0.061814]

Procesando archivo: Albacete.csv
Ciudad en archivo: Albacete
Ciudad en índices: Albacete


Interpolando índices y uniendo datasets:  17%|█▋        | 2/12 [00:09<00:45,  4.60s/it]

...

✅ Guardado: ..\..\datasets\archivos_cont_clima_indices\Valencia.csv
   Filas finales: 105192
   Método NDVI: PCHIP
   Método NDBI: PCHIP
   Rango NDVI: [-0.0107, 0.1844]
   Rango NDBI: [-0.015571, 0.136567]

Procesando archivo: Valladolid.csv
Ciudad en archivo: Valladolid
Ciudad en índices: Valladolid


Interpolando índices y uniendo datasets:  92%|█████████▏| 11/12 [00:48<00:04,  4.50s/it]

✅ Guardado: ..\..\datasets\archivos_cont_clima_indices\Valladolid.csv
   Filas finales: 105192
   Método NDVI: PCHIP
   Método NDBI: PCHIP
   Rango NDVI: [0.0933, 0.3357]
   Rango NDBI: [-0.041504, 0.15987]

Procesando archivo: Zaragoza.csv
Ciudad en archivo: Zaragoza
Ciudad en índices: Zaragoza


Interpolando índices y uniendo datasets: 100%|██████████| 12/12 [00:52<00:00,  4.40s/it]

✅ Guardado: ..\..\datasets\archivos_cont_clima_indices\Zaragoza.csv
   Filas finales: 105192
   Método NDVI: PCHIP
   Método NDBI: PCHIP
   Rango NDVI: [0.0595, 0.2284]
   Rango NDBI: [-0.087994, 0.148716]
